Diagnóstico de los datos crudos de MiBici (2025-10 a 2026-09)

**Objetivo:** conocer la calidad de `data/raw/` antes de limpiar. Este notebook **solo lee**
los archivos: no modifica nada en `data/raw/`. Todas las transformaciones (fechas, duración,
edad) se hacen en memoria.

Revisamos:

1. Columnas y codificación de cada mes
2. Tipos de dato, nulos y duplicados
3. Rango de fechas y viajes fuera de su mes
4. Duración de los viajes
5. Año de nacimiento y género
6. Estaciones contra la nomenclatura
7. Total de filas

Al final se guarda un resumen en `reports/diagnostico_raw.md`.

In [1]:
import pandas as pd

from mibici.config import RAW_DATA_DIR, REPORTS_DIR

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

archivos_viajes = sorted(RAW_DATA_DIR.glob("datos_abiertos_*.csv"))
archivo_nomenclatura = sorted(RAW_DATA_DIR.glob("nomenclatura_*.csv"))[-1]

print(f"{len(archivos_viajes)} archivos de viajes")
print(f"Nomenclatura: {archivo_nomenclatura.name}")

2026-10-19 16:15:11.310 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: /Users/luisgalindez/mlops-tc5061/mibici-demand


12 archivos de viajes
Nomenclatura: nomenclatura_2026_09.csv


In [2]:
def mes_de(archivo):
    """'datos_abiertos_2025_10.csv' -> '2025-10'"""
    return archivo.stem.removeprefix("datos_abiertos_").replace("_", "-")


def detectar_codificacion(archivo):
    """Regresa 'utf-8' si todo el archivo es UTF-8 válido; si no, asumimos 'latin-1'."""
    try:
        archivo.read_bytes().decode("utf-8")
        return "utf-8"
    except UnicodeDecodeError:
        return "latin-1"

# 1. Columnas y codificación por mes

Leemos solo el encabezado de cada archivo y lo comparamos con el del primer mes (referencia).
La codificación se revisa sobre el archivo **completo**: si algún byte no es UTF-8 válido, el
archivo se marca como Latin-1.

In [3]:
filas = []
for archivo in archivos_viajes:
    codificacion = detectar_codificacion(archivo)
    columnas = pd.read_csv(archivo, nrows=0, encoding=codificacion).columns.tolist()
    filas.append({"mes": mes_de(archivo), "codificacion": codificacion, "columnas": columnas})

resumen_columnas = pd.DataFrame(filas)
referencia = resumen_columnas.loc[0, "columnas"]
resumen_columnas["n_columnas"] = resumen_columnas["columnas"].apply(len)
resumen_columnas["igual_a_referencia"] = resumen_columnas["columnas"].apply(lambda c: c == referencia)
resumen_columnas["mismos_nombres_otro_orden"] = resumen_columnas["columnas"].apply(
    lambda c: c != referencia and sorted(c) == sorted(referencia)
)
codificaciones = dict(zip(resumen_columnas["mes"], resumen_columnas["codificacion"]))

print("Columnas de referencia:", referencia)
resumen_columnas.drop(columns="columnas")

Columnas de referencia: ['Viaje_Id', 'Usuario_Id', 'Genero', 'Año_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id']


,mes,codificacion,n_columnas,igual_a_referencia,mismos_nombres_otro_orden
0,2025-10,utf-8,8,True,False
1,2025-11,utf-8,8,True,False
2,2025-12,utf-8,8,True,False
3,2026-01,latin-1,8,True,False
4,2026-02,utf-8,8,True,False
5,2026-03,utf-8,8,True,False
6,2026-04,utf-8,8,True,False
7,2026-05,utf-8,8,True,False
8,2026-06,utf-8,8,True,False
9,2026-07,utf-8,8,True,False


# 2. Tipos de dato, nulos y duplicados

Cargamos los 12 meses con la codificación detectada y dejamos que pandas infiera los tipos.
Agregamos la columna `mes_archivo` para saber de qué archivo viene cada fila.

In [4]:
partes = []
tipos_por_mes = {}
for archivo in archivos_viajes:
    mes = mes_de(archivo)
    df = pd.read_csv(archivo, encoding=codificaciones[mes])
    tipos_por_mes[mes] = df.dtypes.astype(str)
    df["mes_archivo"] = mes
    partes.append(df)

viajes = pd.concat(partes, ignore_index=True)
columnas_originales = referencia
print(f"{len(viajes):,} viajes cargados")

4,497,877 viajes cargados


## 2.1 Tipos inferidos por mes

Si un mes tiene un tipo distinto en alguna columna, es señal de valores raros (texto en una
columna numérica, por ejemplo).

In [5]:
tipos = pd.DataFrame(tipos_por_mes).T
print("¿Mismos tipos en todos los meses?", (tipos.nunique() == 1).all())
tipos

¿Mismos tipos en todos los meses? True


,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
2025-10,int64,int64,str,float64,str,str,int64,int64
2025-11,int64,int64,str,float64,str,str,int64,int64
2025-12,int64,int64,str,float64,str,str,int64,int64
2026-01,int64,int64,str,float64,str,str,int64,int64
2026-02,int64,int64,str,float64,str,str,int64,int64
2026-03,int64,int64,str,float64,str,str,int64,int64
2026-04,int64,int64,str,float64,str,str,int64,int64
2026-05,int64,int64,str,float64,str,str,int64,int64
2026-06,int64,int64,str,float64,str,str,int64,int64
2026-07,int64,int64,str,float64,str,str,int64,int64


## 2.2 Nulos por columna y por mes

In [6]:
nulos_por_mes = viajes[columnas_originales].isna().groupby(viajes["mes_archivo"]).sum()
nulos_por_mes.loc["TOTAL"] = nulos_por_mes.sum()
nulos_por_mes

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
mes_archivo,,,,,,,,
2025-10,0,0,4343,33,0,0,0,0
2025-11,0,0,3894,65,0,0,0,0
2025-12,0,0,2791,105,0,0,0,0
2026-01,0,0,2616,90,0,0,0,0
2026-02,0,0,2400,41,0,0,0,0
2026-03,0,0,2470,39,0,0,0,0
2026-04,0,0,2203,22,0,0,0,0
2026-05,0,0,1704,8,0,0,0,0
2026-06,0,0,2283,10,0,0,0,0


## 2.3 Filas duplicadas

- **Completas:** todas las columnas originales iguales (dentro o entre archivos).
- **Por `Viaje_Id`:** el mismo ID de viaje aparece más de una vez.

In [7]:
duplicadas_completas = viajes.duplicated(subset=columnas_originales).sum()
id_repetido = viajes["Viaje_Id"].duplicated(keep=False)
ids_repetidos = viajes.loc[id_repetido, "Viaje_Id"].nunique()
filas_id_repetido = id_repetido.sum()
# IDs que aparecen en más de un archivo
ids_entre_meses = (viajes.groupby("Viaje_Id")["mes_archivo"].nunique() > 1).sum()

print(f"Filas duplicadas completas: {duplicadas_completas:,}")
print(f"Filas con Viaje_Id repetido: {filas_id_repetido:,} ({ids_repetidos:,} IDs distintos)")
print(f"IDs que aparecen en más de un archivo: {ids_entre_meses:,}")

Filas duplicadas completas: 758
Filas con Viaje_Id repetido: 1,516 (758 IDs distintos)
IDs que aparecen en más de un archivo: 0


In [8]:
# ¿En qué meses están los duplicados?
duplicadas_por_mes = viajes.duplicated(subset=columnas_originales).groupby(viajes["mes_archivo"]).sum()
duplicadas_por_mes[duplicadas_por_mes > 0].to_frame("filas_duplicadas")

,filas_duplicadas
mes_archivo,
2026-09,758


In [9]:
# Ejemplos de Viaje_Id repetido (si hay)
viajes[id_repetido].sort_values("Viaje_Id").head(10)

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,mes_archivo
4007953,45705948,4610130,M,2001.0,2026-09-01 05:42:47,2026-09-01 06:02:02,265,303,2026-09
4007954,45705948,4610130,M,2001.0,2026-09-01 05:42:47,2026-09-01 06:02:02,265,303,2026-09
4008775,45706846,4610130,M,2001.0,2026-09-01 07:11:03,2026-09-01 07:29:32,303,265,2026-09
4008776,45706846,4610130,M,2001.0,2026-09-01 07:11:03,2026-09-01 07:29:32,303,265,2026-09
4009183,45707318,1110472,F,1992.0,2026-09-01 07:37:33,2026-09-01 07:42:47,102,105,2026-09
4009184,45707318,1110472,F,1992.0,2026-09-01 07:37:33,2026-09-01 07:42:47,102,105,2026-09
4009203,45707337,4610243,M,1972.0,2026-09-01 07:38:39,2026-09-01 07:51:52,363,445,2026-09
4009204,45707337,4610243,M,1972.0,2026-09-01 07:38:39,2026-09-01 07:51:52,363,445,2026-09
4009524,45707670,1163105,F,1987.0,2026-09-01 07:53:10,2026-09-01 08:18:27,434,217,2026-09
4009525,45707670,1163105,F,1987.0,2026-09-01 07:53:10,2026-09-01 08:18:27,434,217,2026-09


# 3. Rango de fechas por archivo

Convertimos `Inicio_del_viaje` y `Fin_del_viaje` a fecha. Si algún valor no tiene el formato
`AAAA-MM-DD HH:MM:SS`, queda como `NaT` y lo contamos. Después revisamos si hay viajes que
**inician** fuera del mes del archivo (el fin puede caer en el mes siguiente si el viaje
cruza la medianoche del último día).

In [10]:
for columna in ["Inicio_del_viaje", "Fin_del_viaje"]:
    convertida = pd.to_datetime(viajes[columna], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    sin_formato = (convertida.isna() & viajes[columna].notna()).sum()
    print(f"{columna}: {sin_formato:,} valores con formato no reconocido")
    viajes[columna] = convertida

Inicio_del_viaje: 0 valores con formato no reconocido


Fin_del_viaje: 0 valores con formato no reconocido


In [11]:
mes_inicio = viajes["Inicio_del_viaje"].dt.to_period("M").astype(str)
mes_fin = viajes["Fin_del_viaje"].dt.to_period("M").astype(str)
viajes["inicio_fuera_de_mes"] = mes_inicio != viajes["mes_archivo"]
viajes["fin_fuera_de_mes"] = mes_fin != viajes["mes_archivo"]

rango_fechas = viajes.groupby("mes_archivo").agg(
    inicio_min=("Inicio_del_viaje", "min"),
    inicio_max=("Inicio_del_viaje", "max"),
    fin_min=("Fin_del_viaje", "min"),
    fin_max=("Fin_del_viaje", "max"),
    inicio_fuera_de_mes=("inicio_fuera_de_mes", "sum"),
    fin_fuera_de_mes=("fin_fuera_de_mes", "sum"),
)
rango_fechas

,inicio_min,inicio_max,fin_min,fin_max,inicio_fuera_de_mes,fin_fuera_de_mes
mes_archivo,,,,,,
2025-10,2025-10-01 00:00:26,2025-10-31 23:59:57,2025-10-01 00:07:03,2025-11-01 03:41:08,0,24
2025-11,2025-11-01 00:00:07,2025-11-30 23:59:53,2025-11-01 00:03:03,2025-12-01 00:22:10,0,16
2025-12,2025-12-01 00:01:03,2025-12-31 23:59:08,2025-12-01 00:08:37,2026-01-01 00:09:47,0,7
2026-01,2026-01-01 00:03:07,2026-01-31 23:58:47,2026-01-01 00:03:39,2026-02-01 00:23:18,0,21
2026-02,2026-02-01 00:00:01,2026-02-28 23:57:41,2026-02-01 00:02:02,2026-03-02 18:13:28,0,21
2026-03,2026-03-01 00:00:09,2026-03-31 23:59:02,2026-03-01 00:09:27,2026-04-01 00:19:29,0,21
2026-04,2026-04-01 00:00:26,2026-04-30 23:59:03,2026-04-01 00:00:47,2026-05-01 12:22:39,0,42
2026-05,2026-05-01 00:01:04,2026-05-31 23:59:19,2026-05-01 00:07:48,2026-06-01 07:42:25,0,15
2026-06,2026-06-01 00:06:46,2026-06-30 23:59:58,2026-06-01 00:09:53,2026-07-01 00:48:48,0,75


In [12]:
# Ejemplos de viajes que inician fuera del mes de su archivo (si hay)
viajes.loc[viajes["inicio_fuera_de_mes"], columnas_originales + ["mes_archivo"]].head(10)

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,mes_archivo


# 4. Duración de los viajes

`duracion_min = Fin_del_viaje - Inicio_del_viaje` en minutos. Buscamos duraciones imposibles
(negativas o cero), muy cortas (menos de 1 minuto, típicamente bicicletas que se sacan y se
regresan a la misma estación) y los percentiles altos para decidir el corte de outliers.

In [13]:
viajes["duracion_min"] = (
    viajes["Fin_del_viaje"] - viajes["Inicio_del_viaje"]
).dt.total_seconds() / 60
duracion = viajes["duracion_min"]
misma_estacion = viajes["Origen_Id"] == viajes["Destino_Id"]

conteo_duracion = pd.Series(
    {
        "negativa": (duracion < 0).sum(),
        "cero": (duracion == 0).sum(),
        "mayor a 0 y menor a 1 min": ((duracion > 0) & (duracion < 1)).sum(),
        "menor a 1 min y misma estación": ((duracion < 1) & misma_estacion).sum(),
        "mayor a 1 h": (duracion > 60).sum(),
        "mayor a 3 h": (duracion > 180).sum(),
        "mayor a 24 h": (duracion > 24 * 60).sum(),
    },
    name="viajes",
).to_frame()
conteo_duracion["porcentaje"] = (conteo_duracion["viajes"] / len(viajes) * 100).round(3)
conteo_duracion

,viajes,porcentaje
negativa,0,0.000
cero,3791,0.084
mayor a 0 y menor a 1 min,153899,3.422
menor a 1 min y misma estación,141053,3.136
mayor a 1 h,4356,0.097
mayor a 3 h,510,0.011
mayor a 24 h,13,0.000


In [14]:
percentiles_duracion = duracion.quantile([0.5, 0.9, 0.95, 0.99, 0.999]).rename(
    lambda p: f"p{p * 100:g}"
)
percentiles_duracion["máximo"] = duracion.max()
percentiles_duracion.round(2).to_frame("minutos")

,minutos
p50,9.35
p90,20.72
p95,24.47
p99,33.43
p99.9,59.18
máximo,5920.05


In [15]:
# Viajes con duración negativa (si hay)
viajes.loc[duracion < 0, columnas_originales + ["duracion_min"]].head(10)

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,duracion_min


# 5. Año de nacimiento y género

Calculamos la edad al momento del viaje (`año del viaje - año de nacimiento`). Como referencia
provisional marcamos como sospechosas las edades **menores a 16** o **mayores a 90**; el corte
definitivo se decide en la limpieza.

In [16]:
anio = viajes["Año_de_nacimiento"]
edad = viajes["Inicio_del_viaje"].dt.year - anio

print(anio.describe().round(1))
conteo_edad = pd.Series(
    {
        "año nulo": anio.isna().sum(),
        "edad menor a 16": (edad < 16).sum(),
        "edad mayor a 90": (edad > 90).sum(),
    },
    name="viajes",
).to_frame()
conteo_edad["porcentaje"] = (conteo_edad["viajes"] / len(viajes) * 100).round(3)
conteo_edad

count    4497343.0
mean        1990.5
std           18.8
min          199.0
25%         1985.0
50%         1993.0
75%         1999.0
max         2010.0
Name: Año_de_nacimiento, dtype: float64


,viajes,porcentaje
año nulo,534,0.012
edad menor a 16,0,0.000
edad mayor a 90,319,0.007


In [17]:
# Años de nacimiento más extremos y cuántos viajes tienen cada uno
conteo_anios = anio.value_counts().sort_index()
print("Más antiguos:")
print(conteo_anios.head(10).to_string())
print("\nMás recientes:")
print(conteo_anios.tail(10).to_string())

Más antiguos:
Año_de_nacimiento
199.0      316
1420.0       3
1939.0     255
1941.0     307
1942.0    1065
1943.0       1
1944.0      23
1945.0     510
1946.0     130
1947.0      37

Más recientes:
Año_de_nacimiento
2001.0    208855
2002.0    174620
2003.0    131001
2004.0    103118
2005.0     72990
2006.0     62818
2007.0     44734
2008.0     17356
2009.0      4465
2010.0       918


In [18]:
conteo_genero = viajes["Genero"].value_counts(dropna=False).to_frame("viajes")
conteo_genero["porcentaje"] = (conteo_genero["viajes"] / len(viajes) * 100).round(2)
conteo_genero

,viajes,porcentaje
Genero,,
M,3241248,72.06
F,1223647,27.20
NaN,32982,0.73


# 6. Estaciones contra la nomenclatura

Primero revisamos la nomenclatura (IDs duplicados, nulos, capacidad). Después buscamos
estaciones que aparecen en los viajes pero no en la nomenclatura, y estaciones de la
nomenclatura sin ningún viaje en el periodo. **Ojo:** la nomenclatura es una foto de un solo
mes, así que una estación que ya se dio de baja puede no aparecer.

In [19]:
nomenclatura = pd.read_csv(archivo_nomenclatura, encoding=detectar_codificacion(archivo_nomenclatura))
print(f"{len(nomenclatura)} estaciones, columnas: {nomenclatura.columns.tolist()}")
print(f"IDs duplicados: {nomenclatura['id'].duplicated().sum()}")
print(f"Capacidad (dpcapacity) igual a 0: {(nomenclatura['dpcapacity'] == 0).sum()}")
print("Nulos por columna:")
print(nomenclatura.isna().sum().to_string())
nomenclatura.describe().round(4)

484 estaciones, columnas: ['id', 'name', 'latitude', 'longitude', 'dpcapacity']
IDs duplicados: 0
Capacidad (dpcapacity) igual a 0: 4
Nulos por columna:
id            0
name          0
latitude      0
longitude     0
dpcapacity    0


,id,latitude,longitude,dpcapacity
count,484.0000,484.0000,484.0000,484.0000
mean,254.7583,20.6785,-103.3619,16.8822
std,148.2938,0.0220,0.0316,4.1269
min,2.0000,20.6361,-103.4332,0.0000
25%,129.7500,20.6651,-103.3842,15.0000
50%,252.0000,20.6744,-103.3624,15.0000
75%,379.2500,20.6886,-103.3393,19.0000
max,519.0000,20.7395,-103.2971,39.0000


In [20]:
ids_nomenclatura = set(nomenclatura["id"])
origen_faltante = viajes.loc[~viajes["Origen_Id"].isin(ids_nomenclatura), "Origen_Id"]
destino_faltante = viajes.loc[~viajes["Destino_Id"].isin(ids_nomenclatura), "Destino_Id"]

estaciones_faltantes = (
    pd.DataFrame(
        {
            "viajes_como_origen": origen_faltante.value_counts(),
            "viajes_como_destino": destino_faltante.value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
    .sort_index()
)
estaciones_faltantes.index.name = "estacion_id"
print(f"Estaciones en viajes que no están en la nomenclatura: {len(estaciones_faltantes)}")
print(f"Viajes afectados: {(origen_faltante.size + destino_faltante.size):,} referencias "
      f"({(~viajes['Origen_Id'].isin(ids_nomenclatura) | ~viajes['Destino_Id'].isin(ids_nomenclatura)).sum():,} viajes)")
estaciones_faltantes

Estaciones en viajes que no están en la nomenclatura: 2
Viajes afectados: 4,083 referencias (3,954 viajes)


,viajes_como_origen,viajes_como_destino
estacion_id,,
405,3,3
407,1957,2120


In [21]:
# ¿En qué meses aparecen las estaciones que faltan en la nomenclatura?
usa_faltante = ~viajes["Origen_Id"].isin(ids_nomenclatura) | ~viajes["Destino_Id"].isin(ids_nomenclatura)
faltantes_por_mes = viajes[usa_faltante].groupby("mes_archivo").size().to_frame("viajes_con_estacion_faltante")
faltantes_por_mes

,viajes_con_estacion_faltante
mes_archivo,
2026-06,1980
2026-07,1974


In [22]:
ids_con_viajes = set(viajes["Origen_Id"]) | set(viajes["Destino_Id"])
estaciones_sin_viajes = nomenclatura[~nomenclatura["id"].isin(ids_con_viajes)]
es_baja = nomenclatura["name"].str.startswith("ex-")
print(f"Estaciones de la nomenclatura sin viajes en el periodo: {len(estaciones_sin_viajes)}")
print(f"Estaciones con prefijo 'ex-' (dadas de baja) en la nomenclatura: {es_baja.sum()}")
print(f"  de ellas, sin viajes en el periodo: {estaciones_sin_viajes['name'].str.startswith('ex-').sum()}")
print(f"  de ellas, con viajes en el periodo: {(es_baja & nomenclatura['id'].isin(ids_con_viajes)).sum()}")
estaciones_sin_viajes

Estaciones de la nomenclatura sin viajes en el periodo: 15
Estaciones con prefijo 'ex-' (dadas de baja) en la nomenclatura: 14
  de ellas, sin viajes en el periodo: 14
  de ellas, con viajes en el periodo: 0


,id,name,latitude,longitude,dpcapacity
19,22,ex-(GDL-020) Juan Manuel / C.General Coronado,20.678714,-103.365730,15
92,97,ex-(ZPN-007) C. Libertad / Nicolás Bravo,20.726196,-103.392672,19
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,20.685108,-103.333746,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,20.723221,-103.385583,0
108,113,ex-(ZPN-023) C. Juan Diego / C. Sarcófago,20.724890,-103.383614,11
109,114,ex-(ZPN-024) Manuel H. Alatorre / C. Industria,20.723560,-103.381901,11
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,20.686920,-103.334670,0
129,138,ex-(ZPN-044),20.697050,-103.363220,0
138,147,ex-(TLQ-006) C. Zalatitán / Av. Francisco Silva,20.641848,-103.302600,15
140,149,ex-(TLQ-008) C. Juárez / C.Francisco de Miranda,20.638410,-103.313340,15


# 7. Total de filas por mes y en conjunto

In [23]:
filas_por_mes = viajes["mes_archivo"].value_counts().sort_index().to_frame("viajes")
filas_por_mes.loc["TOTAL"] = filas_por_mes["viajes"].sum()
print(f"Usuarios distintos en el periodo: {viajes['Usuario_Id'].nunique():,}")
filas_por_mes

Usuarios distintos en el periodo: 32,562


,viajes
mes_archivo,
2025-10,420064
2025-11,380328
2025-12,335420
2026-01,357180
2026-02,333756
2026-03,376243
2026-04,351851
2026-05,315895
2026-06,355303


# 8. Resumen en `reports/diagnostico_raw.md`

Armamos el reporte con los mismos objetos calculados arriba, así los números del reporte
siempre coinciden con el notebook. Las tablas se escriben en Markdown con una función pequeña
para no agregar dependencias.

In [24]:
def tabla_md(df):
    """Convierte un DataFrame pequeño en una tabla Markdown."""

    def formato(valor):
        if isinstance(valor, bool):
            return "sí" if valor else "no"
        if isinstance(valor, int):
            return f"{valor:,}"
        if isinstance(valor, float):
            return f"{valor:,.2f}"
        return str(valor)

    encabezado = "| " + " | ".join(map(str, df.columns)) + " |"
    separador = "|" + "---|" * len(df.columns)
    filas = ["| " + " | ".join(formato(v) for v in fila) + " |" for fila in df.astype(object).itertuples(index=False)]
    return "\n".join([encabezado, separador, *filas])


# Tabla por mes con lo principal de cada archivo
por_mes = pd.DataFrame(
    {
        "codificación": pd.Series(codificaciones),
        "filas": viajes["mes_archivo"].value_counts(),
        "nulos Genero": nulos_por_mes["Genero"].drop("TOTAL"),
        "nulos Año_de_nacimiento": nulos_por_mes["Año_de_nacimiento"].drop("TOTAL"),
        "duplicadas": duplicadas_por_mes,
        "inicio fuera de mes": rango_fechas["inicio_fuera_de_mes"],
        "fin en mes siguiente": rango_fechas["fin_fuera_de_mes"],
    }
).sort_index()
por_mes.index.name = "mes"

anios_invalidos = conteo_anios[conteo_anios.index < 1920]
total = len(viajes)

In [25]:
reporte = f"""# Diagnóstico de datos crudos — MiBici 2025-10 a 2026-09

Generado por `notebooks/0.1-lg-diagnostico-raw.ipynb` a partir de `data/raw/` (solo lectura).

## Números clave

- **{len(archivos_viajes)} archivos de viajes** + nomenclatura (`{archivo_nomenclatura.name}`, {len(nomenclatura)} estaciones).
- **{total:,} viajes** en total, de **{viajes['Usuario_Id'].nunique():,} usuarios** distintos.
- Las **{len(referencia)} columnas** tienen el mismo nombre, orden y tipo en todos los meses: {", ".join(f"`{c}`" for c in referencia)}.
- Codificación: todos los meses son UTF-8 excepto **{", ".join(m for m, c in codificaciones.items() if c != "utf-8")}** (Latin-1).
- Fechas: 0 valores con formato distinto de `AAAA-MM-DD HH:MM:SS` y **0 viajes que inician fuera del mes de su archivo**.
- Duplicados: **{duplicadas_completas:,} filas duplicadas completas**, todas en {", ".join(duplicadas_por_mes[duplicadas_por_mes > 0].index)}. Cada `Viaje_Id` repetido corresponde a un duplicado exacto ({ids_repetidos:,} IDs, {filas_id_repetido:,} filas); ningún ID se repite entre archivos.

## Por mes

{tabla_md(por_mes.reset_index())}

"Fin en mes siguiente" son viajes que empiezan el último día del mes y terminan después de la medianoche; es esperado.

## Nulos

Solo hay nulos en dos columnas: `Genero` ({nulos_por_mes.loc["TOTAL", "Genero"]:,}, {nulos_por_mes.loc["TOTAL", "Genero"] / total:.2%}) y `Año_de_nacimiento` ({nulos_por_mes.loc["TOTAL", "Año_de_nacimiento"]:,}, {nulos_por_mes.loc["TOTAL", "Año_de_nacimiento"] / total:.3%}). Las columnas de fechas, IDs y estaciones no tienen nulos.

## Duración de los viajes

{tabla_md(conteo_duracion.reset_index().rename(columns={"index": "caso"}))}

{tabla_md(percentiles_duracion.round(2).to_frame("minutos").reset_index().rename(columns={"index": "percentil"}))}

## Año de nacimiento y género

- Rango de años: {int(anio.min())} a {int(anio.max())}. Ninguna edad menor a 16 años.
- Años imposibles (antes de 1920): {", ".join(f"**{int(a)}** ({n:,} viajes)" for a, n in anios_invalidos.items())}. El siguiente año válido es {int(conteo_anios[conteo_anios.index >= 1920].index.min())}.

{tabla_md(conteo_genero.reset_index().fillna("(nulo)"))}

## Estaciones contra la nomenclatura

- La nomenclatura no tiene IDs duplicados ni nulos; **{(nomenclatura["dpcapacity"] == 0).sum()} estaciones con capacidad 0**.
- **{len(estaciones_faltantes)} estaciones** aparecen en viajes pero no en la nomenclatura ({usa_faltante.sum():,} viajes afectados, solo en {", ".join(faltantes_por_mes.index)}; parecen estaciones temporales):

{tabla_md(estaciones_faltantes.reset_index())}

- **{len(estaciones_sin_viajes)} estaciones** de la nomenclatura no tienen viajes en el periodo; {estaciones_sin_viajes["name"].str.startswith("ex-").sum()} tienen el prefijo `ex-` (dadas de baja). La nomenclatura incluye {es_baja.sum()} estaciones `ex-` en total.

{tabla_md(estaciones_sin_viajes[["id", "name", "dpcapacity"]])}

## Propuestas para la limpieza (a decidir en equipo)

1. Leer **{", ".join(m for m, c in codificaciones.items() if c != "utf-8")}** con `encoding="latin-1"` (o detectar la codificación por archivo).
2. Eliminar las **{duplicadas_completas:,} filas duplicadas** exactas.
3. Eliminar duraciones **cero** ({conteo_duracion.loc["cero", "viajes"]:,}). No hay negativas.
4. Viajes de **menos de 1 minuto en la misma estación** ({conteo_duracion.loc["menor a 1 min y misma estación", "viajes"]:,}, {conteo_duracion.loc["menor a 1 min y misma estación", "porcentaje"]:.2f}%): probablemente bicicletas que se sacan y se regresan de inmediato. Proponemos excluirlos porque no representan demanda real.
5. Corte superior de duración: p99.9 = {percentiles_duracion["p99.9"]:.1f} min; solo {conteo_duracion.loc["mayor a 3 h", "viajes"]:,} viajes duran más de 3 h y {conteo_duracion.loc["mayor a 24 h", "viajes"]:,} más de 24 h (máximo {percentiles_duracion["máximo"] / 60:.1f} h). Proponemos un corte en **3 h**. Para pronosticar salidas por estación y hora la duración pesa poco, así que el corte importa más para el EDA.
6. `Año_de_nacimiento` antes de 1920 → nulo (no eliminar el viaje: la salida sí ocurrió).
7. `Genero` nulo → categoría "desconocido".
8. Estaciones {", ".join(map(str, estaciones_faltantes.index))}: no están en la nomenclatura de {archivo_nomenclatura.stem.removeprefix("nomenclatura_").replace("_", "-")}; buscarlas en nomenclaturas anteriores o conservarlas sin coordenadas.
"""

ruta_reporte = REPORTS_DIR / "diagnostico_raw.md"
ruta_reporte.write_text(reporte, encoding="utf-8")
print(f"Reporte guardado en reports/{ruta_reporte.name}")

Reporte guardado en reports/diagnostico_raw.md
